# Crop mapping — annual agricultural features (2021–2024)

DWR land-use features for Salinas HUC8 nitrate wells, extending the original crop-mapping workflow to annual 500 m and 1,000 m buffers. Crop year *t* is joined to groundwater measurements from year *t + 1*. The 2020 map is used only to calculate changes for 2021.

## 0. Crop features for each well — inputs and settings

Uses the same broad DWR crop classes as the earlier mapping. Raw DWR geodatabases are external to Git; by default, the notebook looks for the shared `Data/Erdos Project/raw/Crops` folder alongside `GitHub`. Override `ERDOS_CROPS_DIR` only if your data are stored elsewhere.

In [1]:
from pathlib import Path
import os
import hashlib
import json

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio
from pyproj import Transformer

METRIC_CRS = 3310
HUC8 = "18060005"
RADII = (500, 1000)
YEARS = (2021, 2022, 2023, 2024)
GROUPS = ["veg_berry", "vineyard", "orchard", "grain_field_pasture",
          "idle_fallow", "urban"]
FARMED = ["veg_berry", "vineyard", "orchard", "grain_field_pasture"]
CLASS_GROUP = {
    "T": "veg_berry", "V": "vineyard", "D": "orchard", "C": "orchard",
    "Y": "orchard", "YP": "orchard", "G": "grain_field_pasture",
    "F": "grain_field_pasture", "P": "grain_field_pasture",
    "R": "grain_field_pasture", "I": "idle_fallow", "X": "idle_fallow",
}
CYCLES = {"S": 1, "D": 2, "T": 3, "Q": 4}
EXPECTED_STATUS = {2021: "Final", 2022: "Final", 2023: "Final",
                   2024: "Provisional"}

# Locate the project and external DWR data without machine-specific paths.
REPO_DIR = next(
    (p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "notebooks/groundwater/groundwater_nitrate_clean.csv").is_file()),
    None,
)
if REPO_DIR is None:
    raise FileNotFoundError("Run this notebook inside the cloned project repository.")

WORKSPACE = REPO_DIR.parent.parent
root = Path(os.environ.get(
    "ERDOS_CROPS_DIR",
    str(WORKSPACE / "Data" / "Erdos Project" / "raw" / "Crops"),
)).expanduser().resolve()
output = Path(os.environ.get(
    "ERDOS_CROP_OUTPUT_DIR",
    str(WORKSPACE / "Data" / "Erdos Project" / "processed" / "Crops" / "annual_lag1_notebook_run"),
)).expanduser().resolve()
repo = REPO_DIR

if not root.is_dir():
    raise FileNotFoundError(
        f"DWR input folder not found: {root}. Set ERDOS_CROPS_DIR to your DWR data folder."
    )
print("Repository:", repo)
print("DWR input folder:", root)
print("Output folder (created at end):", output)

Repository: C:\Users\lomel\Documents\Data Science\GitHub\fall-2026-infrastructure-placement
DWR input folder: C:\Users\lomel\Documents\Data Science\Data\Erdos Project\raw\Crops
Output folder (created at end): C:\Users\lomel\Documents\Data Science\Data\Erdos Project\processed\Crops\annual_lag1_notebook_run


## 1. DWR crop maps and classifications

`SYMB_CLASS` codes are grouped as in the original crop mapping, including `T` as `veg_berry`. `MULTIUSE` records the number of crop cycles. The annual inputs are DWR maps from 2020–2024; 2024 is provisional.

In [2]:
def require(ok, message):
    if not ok:
        raise ValueError(message)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def find_gdb(root, token):
    matches = [p.parent for p in root.rglob("a00000001.gdbtable")
               if token.lower() in p.parent.name.lower()]
    require(len(matches) == 1, f"Expected one GDB for {token}; found {matches}")
    return matches[0]


def read_crop_map(path, boundary):
    layer = pyogrio.list_layers(path)[0, 0]
    info = pyogrio.read_info(path, layer=layer)
    require(info["crs"] is not None, f"Missing CRS: {path}")
    required = {"UniqueID", "SYMB_CLASS", "MULTIUSE", "COUNTY"}
    require(required <= set(info["fields"]), f"Unexpected source schema: {path}")
    xmin, ymin, xmax, ymax = boundary.total_bounds
    to_source = Transformer.from_crs(METRIC_CRS, info["crs"], always_xy=True)
    bbox = to_source.transform_bounds(xmin - 1001, ymin - 1001,
                                      xmax + 1001, ymax + 1001,
                                      densify_pts=41)
    data = pyogrio.read_dataframe(path, layer=layer, bbox=bbox, fid_as_index=True)
    require(data.index.is_unique and data.UniqueID.notna().all()
            and data.UniqueID.is_unique, f"Duplicate/missing source IDs: {path}")
    require(data.geometry.notna().all() and (~data.geometry.is_empty).all(),
            f"Missing source geometry: {path}")
    data.geometry = data.geometry.force_2d().to_crs(METRIC_CRS).buffer(0)
    require(data.is_valid.all(), f"Invalid geometry after repair: {path}")
    letters = (data.SYMB_CLASS.astype("string").str.split("|").str[0]
               .str.strip().str.upper())
    unknown = sorted(set(letters.dropna()) - set(CLASS_GROUP) - {"U", "UL"})
    require(not unknown and letters.notna().all(),
            f"Unreviewed DWR crop classes in {path}: {unknown}")
    data["crop_group"] = letters.map(CLASS_GROUP)
    data.loc[letters.str.startswith("U", na=False), "crop_group"] = "urban"
    data["crop_cycles"] = data.MULTIUSE.astype("string").str.strip().map(CYCLES)
    require(data.crop_cycles.notna().all(), f"Unreviewed MULTIUSE values: {path}")
    data["source_fid"] = data.index
    data = data.reset_index(drop=True)
    status = (sorted(data.DataStatus.dropna().astype(str).unique())
              if "DataStatus" in data else ["Not provided for 2020"])
    return data, {"path": str(path), "layer": layer, "crs": str(info["crs"]),
                  "statewide_feature_count_metadata_only": int(info["features"]),
                  "regional_bbox_rows_loaded": len(data), "source_bbox": list(bbox),
                  "regional_data_status": status}


## 2. Buffer intersections and agricultural features

For every map year, calculate both 500 m and 1,000 m features. Full buffers can cross county boundaries. Crop-group percentages use the theoretical circular area, while mapped-polygon footprint uses the actual buffer area. Unmapped area is not assumed to be nonagricultural land.

In [3]:
def summarize(points, candidates, radius):
    circles = points[["gm_well_id", "geometry"]].copy()
    circles.geometry = circles.buffer(radius, quad_segs=16)
    grouped = candidates.dissolve(by=["crop_group", "crop_cycles"]).reset_index()
    pieces = gpd.overlay(circles, grouped[["crop_group", "crop_cycles", "geometry"]],
                         how="intersection", keep_geom_type=True)
    pieces["area_m2"] = pieces.area
    ids = pd.Index(points.gm_well_id.astype("string"), name="gm_well_id")
    areas = pieces.pivot_table(index="gm_well_id", columns="crop_group",
                               values="area_m2", aggfunc="sum")
    areas = areas.reindex(index=ids, columns=GROUPS, fill_value=0).fillna(0)
    mapped = pieces.groupby("gm_well_id").geometry.apply(lambda s: s.union_all().area)
    mapped = mapped.reindex(ids, fill_value=0)
    summed = areas.sum(axis=1)
    overlap = (summed - mapped).clip(lower=0)
    require(float(overlap.max()) < 1.0, "Mapped classes overlap by >= 1 m2.")
    circle_area = np.pi * radius**2
    buffer_area = pd.Series(circles.area.to_numpy(), index=ids)
    require((mapped <= buffer_area + 1e-4).all(), "Mapped area exceeds buffer.")
    require(np.allclose(buffer_area, circle_area * 0.9983943930356184,
                        rtol=1e-8), "Unexpected buffer units/resolution.")

    table = pd.DataFrame(index=ids)
    for group in GROUPS:
        table[f"pct_{group}_{radius}m"] = areas[group] / circle_area * 100
        # This is composition inside mapped polygons, not a whole-buffer share.
        table[f"pct_{group}_of_mapped_{radius}m"] = (
            areas[group] / mapped.replace(0, np.nan) * 100)
    table[f"pct_cropland_{radius}m"] = areas[FARMED].sum(axis=1) / circle_area * 100
    table[f"pct_cropland_of_mapped_{radius}m"] = (
        areas[FARMED].sum(axis=1) / mapped.replace(0, np.nan) * 100)
    farm = pieces[pieces.crop_group.isin(FARMED)]
    farm_area = farm.groupby("gm_well_id").area_m2.sum().reindex(ids, fill_value=0)
    weighted = (farm.area_m2 * farm.crop_cycles).groupby(farm.gm_well_id).sum()
    table[f"crop_cycles_{radius}m"] = weighted.reindex(ids) / farm_area.replace(0, np.nan)
    multi = farm[farm.crop_cycles.ge(2)].groupby("gm_well_id").area_m2.sum()
    table[f"pct_multicrop_{radius}m"] = multi.reindex(ids, fill_value=0) / circle_area * 100
    table[f"buffer_area_m2_{radius}m"] = buffer_area
    table[f"dwr_polygon_area_m2_{radius}m"] = mapped
    table[f"dwr_polygon_footprint_pct_{radius}m"] = mapped / buffer_area * 100
    table[f"outside_dwr_polygons_pct_{radius}m"] = (
        (buffer_area - mapped).clip(lower=0) / buffer_area * 100)
    table[f"dwr_polygon_status_{radius}m"] = np.where(
        mapped.le(1e-8), "no_dwr_polygon", "dwr_polygon_present_completeness_unknown")
    no_map = mapped.le(1e-8)
    crop_columns = [c for c in table if c.startswith("pct_") or c.startswith("crop_cycles_")]
    table.loc[no_map, crop_columns] = np.nan
    percents = table[[c for c in table if c.startswith("pct_")]]
    require(percents.dropna().ge(-1e-8).all().all()
            and percents.dropna().le(100 + 1e-8).all().all(),
            "Percentage out of bounds.")
    require(table[f"crop_cycles_{radius}m"].dropna().between(1, 4).all(),
            "Crop cycles outside 1-4.")
    qa = {"well_count": len(table), "overlay_piece_count": len(pieces),
          "max_overlap_area_m2": float(overlap.max()),
          "dwr_polygon_status_counts": table[f"dwr_polygon_status_{radius}m"].value_counts().to_dict(),
          "dwr_polygon_footprint_pct_summary": table[f"dwr_polygon_footprint_pct_{radius}m"].describe().to_dict()}
    return table, qa


### 2.1. Groundwater wells and study area

In [4]:
require(not output.exists(), f"Output directory exists; choose a new path: {output}")
require(not output.is_relative_to(root), "Do not put outputs inside raw DWR data.")
nitrate_path = repo / "notebooks/groundwater/groundwater_nitrate_clean.csv"
boundary_path = find_gdb(root, "NHD_H_18060005")
boundary = gpd.read_file(boundary_path, layer="WBDHU8",
                         where=f"huc8 = '{HUC8}'").to_crs(METRIC_CRS)
require(len(boundary) == 1 and boundary.is_valid.all(), "Unexpected HUC8 boundary.")
boundary_shape = boundary.geometry.union_all()
obs = pd.read_csv(nitrate_path, dtype={"gm_well_id": "string"},
                  parse_dates=["gm_samp_collection_date"])
require(len(obs) == 5280 and obs.gm_well_id.nunique() == 1993,
        "Nitrate input changed; review before processing.")
require(not obs.duplicated(["gm_well_id", "gm_samp_collection_date"]).any(),
        "Duplicate well/date keys in nitrate input.")
obs["measurement_year"] = obs.gm_samp_collection_date.dt.year
require(set(obs.measurement_year.unique()) == {2022, 2023, 2024, 2025},
        "Unexpected measurement years.")
point_rows = obs[["gm_well_id", "gm_latitude", "gm_longitude"]].drop_duplicates()
require(point_rows.gm_well_id.is_unique, "Well has multiple coordinate locations.")
all_points = gpd.GeoDataFrame(
    point_rows, geometry=gpd.points_from_xy(point_rows.gm_longitude, point_rows.gm_latitude),
    crs=4326).to_crs(METRIC_CRS)
require(all_points.geometry.covered_by(boundary_shape).all(), "A nitrate well lies outside HUC8.")
hashes = {str(p): sha256(p) for p in [nitrate_path,
          repo / "notebooks/crop_mapping/wells_nitrate.csv",
          repo / "notebooks/crop_mapping/well_crop_features_all.csv"]}

### 2.2. Crop features for each DWR map year

This cell performs the buffer and polygon intersections; it can take a while with the original DWR geodatabases.

In [5]:
map_features, map_qa, source_versions = {}, {}, {}
all_support = all_points.buffer(max(RADII), quad_segs=16).union_all()
for year in range(min(YEARS) - 1, max(YEARS) + 1):
    source_path = find_gdb(root, str(year))
    crop_map, source_versions[str(year)] = read_crop_map(source_path, boundary)
    if year in EXPECTED_STATUS:
        require(source_versions[str(year)]["regional_data_status"] == [EXPECTED_STATUS[year]],
                f"Unexpected DWR DataStatus for {year}: {source_versions[str(year)]}")
    candidates = crop_map[crop_map.intersects(all_support)].copy()
    del crop_map
    yearly_tables = []
    map_qa[str(year)] = {"candidate_polygons_touching_any_well_buffer": len(candidates)}
    for radius in RADII:
        table, qa = summarize(all_points, candidates, radius)
        yearly_tables.append(table)
        map_qa[str(year)][f"{radius}m"] = qa
    map_features[year] = pd.concat(yearly_tables, axis=1)
    del candidates, yearly_tables

C:\Users\lomel\miniforge3\envs\erdos_ds_environment\Lib\site-packages\pyogrio\core.py:169: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))
C:\Users\lomel\miniforge3\envs\erdos_ds_environment\Lib\site-packages\pyogrio\core.py:320: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read_info(
C:\Users\lomel\miniforge3\envs\erdos_ds_environment\Lib\site-packages\pyogrio\raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(
C:\Users\lomel\miniforge3\envs\erdos_ds_environment\Lib\site-packages\pyogrio\core.py:169: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPoly

## 3. Historical land use and one-year lag

For nitrate measurements in 2022–2025, use DWR crop years 2021–2024. Annual changes compare each crop map with the preceding map; 2020 is used only as a baseline for 2021 changes. This is a retrospective modeling assumption, not an estimate of groundwater travel time.

In [6]:
feature_rows, qa_years = [], {}
for year in YEARS:
    measurement_year = year + 1
    wells = obs.loc[obs.measurement_year.eq(measurement_year),
                    ["gm_well_id", "gm_latitude", "gm_longitude"]].drop_duplicates("gm_well_id")
    ids = wells.gm_well_id.astype("string")
    current = map_features[year].loc[ids].copy()
    previous = map_features[year - 1].loc[ids].copy().add_suffix("__previous")
    table = pd.concat([current, previous], axis=1)
    table.insert(0, "gm_well_id", table.index.astype("string"))
    table.insert(1, "crop_year", year)
    table.insert(2, "measurement_year", measurement_year)
    table.insert(3, "dwr_map_status", EXPECTED_STATUS.get(year, "Not provided by source"))
    table.insert(4, "change_base_year", year - 1)
    year_qa = {"measurement_year": measurement_year, "crop_year": year,
               "well_count": len(ids), "rows": len(table),
               "current_map_validation": map_qa[str(year)]}
    for radius in RADII:
        status = (table[f"dwr_polygon_area_m2_{radius}m"].gt(0)
                  & table[f"dwr_polygon_area_m2_{radius}m__previous"].gt(0))
        table[f"annual_change_status_{radius}m"] = np.where(
            status, "both_years_have_polygons_completeness_unknown",
            "no_polygon_in_one_or_both_years")
        for group in [*GROUPS, "cropland", "multicrop"]:
            col = (f"pct_{group}_{radius}m" if group != "multicrop"
                   else f"pct_multicrop_{radius}m")
            delta = f"delta_{group}_pp_{radius}m"
            table[delta] = table[col] - table[f"{col}__previous"]
            table.loc[~status, delta] = np.nan
        col = f"dwr_polygon_footprint_pct_{radius}m"
        table[f"delta_dwr_polygon_footprint_pp_{radius}m"] = (
            table[col] - table[f"{col}__previous"])
        table.loc[~status, f"delta_dwr_polygon_footprint_pp_{radius}m"] = np.nan
    require(table.gm_well_id.is_unique and len(table) == len(ids),
            f"Annual well rows/IDs changed: {year}")
    qa_years[str(year)] = year_qa
    feature_rows.append(table.reset_index(drop=True))

## 4. Quality checks and comparison with 2023 features

Compare equivalent 2023 outputs with the earlier well-crop feature table using its original location and rounding conventions. Do not interpret unmatched areas as zero land use.

In [7]:
def compare_gavin(repo, crops_root):
    observations = pd.read_csv(repo / "notebooks/groundwater/groundwater_nitrate_clean.csv",
                               dtype={"gm_well_id": "string"},
                               parse_dates=["gm_samp_collection_date"])
    wells_2024 = observations[observations.gm_samp_collection_date.dt.year.eq(2024)].gm_well_id.unique()
    expected = pd.read_csv(repo / "notebooks/crop_mapping/well_crop_features_all.csv",
                           dtype={"well_id": "string"})
    legacy_wells = pd.read_csv(repo / "notebooks/crop_mapping/wells_nitrate.csv",
                               dtype={"well_id": "string"})
    ids = set(wells_2024) & set(expected.well_id) & set(legacy_wells.well_id)
    legacy_wells = legacy_wells[legacy_wells.well_id.isin(ids)].copy()
    points = gpd.GeoDataFrame(
        legacy_wells[["well_id", "x_m", "y_m"]].rename(columns={"well_id": "gm_well_id"}),
        geometry=gpd.points_from_xy(legacy_wells.x_m, legacy_wells.y_m), crs=METRIC_CRS)
    path = find_gdb(crops_root, "2023")
    info = pyogrio.read_info(path, layer=pyogrio.list_layers(path)[0, 0])
    xmin, ymin, xmax, ymax = points.total_bounds
    bbox = Transformer.from_crs(METRIC_CRS, info["crs"], always_xy=True).transform_bounds(
        xmin - 1001, ymin - 1001, xmax + 1001, ymax + 1001, densify_pts=41)
    layer = pyogrio.list_layers(path)[0, 0]
    map_data = pyogrio.read_dataframe(path, layer=layer, bbox=bbox)
    map_data.geometry = map_data.geometry.force_2d().to_crs(METRIC_CRS).buffer(0)
    map_data = map_data[map_data.COUNTY.eq("Monterey")].copy()
    letters = map_data.SYMB_CLASS.astype("string").str.split("|").str[0].str.strip().str.upper()
    map_data["crop_group"] = letters.map(CLASS_GROUP)
    map_data.loc[letters.str.startswith("U", na=False), "crop_group"] = "urban"
    map_data["crop_cycles"] = map_data.MULTIUSE.astype("string").str.strip().map(CYCLES).fillna(1)
    map_data = map_data[map_data.crop_group.notna()]
    legacy = expected.set_index("well_id")
    output = []
    for radius in RADII:
        table, _ = summarize(points, map_data, radius)
        for well_id, row in table.iterrows():
            for col, value in row.items():
                if not (col.startswith("pct_") and "_of_mapped_" not in col) and not col.startswith("crop_cycles_"):
                    continue
                feature = col.replace(f"_{radius}m", f"_{radius}m_2023")
                old = legacy.loc[well_id, feature]
                # Gavin rounded saved features to 3 decimals; these locations and
                # his Monterey filter reproduce those legacy calculation inputs.
                match = (pd.isna(value) and pd.isna(old)) or np.isclose(
                    value, old, atol=0.000501, rtol=0)
                output.append({"well_id": well_id, "feature": feature,
                               "recomputed_at_gavin_coordinates": value,
                               "gavin_saved_rounded_3dp": old,
                               "absolute_difference": abs(value - old) if pd.notna(value) and pd.notna(old) else np.nan,
                               "matches_rounding_tolerance": bool(match)})
    result = pd.DataFrame(output)
    require(result.matches_rounding_tolerance.all(), "Gavin 2023 validation failed.")
    return result


In [8]:
output_table = pd.concat(feature_rows, ignore_index=True)
require(not output_table.duplicated(["gm_well_id", "crop_year"]).any(),
        "Duplicate annual feature keys.")
require(len(output_table) == 5178, f"Unexpected annual feature row count: {len(output_table)}")
comparison = compare_gavin(repo, root)
require(hashes == {str(p): sha256(p) for p in [nitrate_path,
        repo / "notebooks/crop_mapping/wells_nitrate.csv",
        repo / "notebooks/crop_mapping/well_crop_features_all.csv"]},
        "An input changed during the run.")

C:\Users\lomel\miniforge3\envs\erdos_ds_environment\Lib\site-packages\pyogrio\core.py:169: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))
C:\Users\lomel\miniforge3\envs\erdos_ds_environment\Lib\site-packages\pyogrio\core.py:320: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read_info(
C:\Users\lomel\miniforge3\envs\erdos_ds_environment\Lib\site-packages\pyogrio\core.py:169: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))
C:\Users\lomel\miniforge3\envs\erdos_ds_environment\Lib\site-packages\pyogrio\raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measur

## 5. Output — annual features and QA

The generated files include annual features, a feature dictionary, the original 2023 comparison, DWR source versions, and QA results. Output goes to a **new directory**; an existing directory is never overwritten.

In [12]:
dictionary = []
for col in output_table.columns:
    if col in {"gm_well_id"}:
        unit, description = "ID", "Groundwater well ID; keep as text."
    elif col.endswith("year"):
        unit, description = "year", "Water/crop year key; not a map publication date."
    elif "status" in col or col == "dwr_map_status":
        unit, description = "category", "Map status; polygon presence does not establish completeness."
    elif "area_m2" in col:
        unit, description = "m2", "Projected EPSG:3310 area; entire buffer, never county-clipped."
    elif col.startswith("delta_"):
        unit, description = "percentage points", "Current DWR map minus preceding crop map."
    elif "cycles" in col:
        unit, description = "cycles/year", "Cropland-area-weighted DWR multiuse cycle count."
    elif "_of_mapped_" in col:
        unit, description = "percent of mapped polygon area", "Conditional share within the union of DWR polygons; not whole-buffer coverage."
    elif "footprint_pct" in col:
        unit, description = "percent of actual buffer", "DWR polygon footprint share only; not map or survey completeness."
    elif "outside_dwr_polygons_pct" in col:
        unit, description = "percent of actual buffer", "Buffer area outside any DWR polygon; land use is unknown from this layer."
    elif col.startswith("pct_"):
        unit, description = "percent of theoretical pi*r^2", "Mapped class area over full circular buffer; 0 is no mapped class intersection, not verified absence."
    else:
        unit, description = "text", "DWR source/version or feature field."
    dictionary.append({"column": col, "unit": unit,
                       "dtype": str(output_table[col].dtype), "description": description})


# Create a unique output folder to preserve previous results.
from datetime import datetime

base_output = output
stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
output = base_output.parent / f"{base_output.name}_{stamp}"
suffix = 1
while output.exists():
    output = base_output.parent / f"{base_output.name}_{stamp}_{suffix}"
    suffix += 1
output.mkdir(parents=True, exist_ok=False)

csv_path = output / "crop_features_lag1_2021_2024.csv"
output_table.to_csv(csv_path, index=False)
pd.DataFrame(dictionary).to_csv(output / "crop_feature_dictionary.csv", index=False)
comparison.to_csv(output / "gavin_2023_validation_all_shared_wells.csv", index=False)
pd.DataFrame([{"map_year": k, **v} for k, v in source_versions.items()]).to_csv(
    output / "dwr_source_versions.csv", index=False)
reread = pd.read_csv(csv_path, dtype={"gm_well_id": "string"})
require(len(reread) == len(output_table)
        and not reread.duplicated(["gm_well_id", "crop_year"]).any(),
        "CSV round-trip failed.")
require(reread.gm_well_id.notna().all(), "Blank IDs after CSV round-trip.")
qa = {"huc8": HUC8, "metric_crs": "EPSG:3310", "radii_m": RADII,
      "measurement_year_to_crop_year": {"2022": 2021, "2023": 2022,
                                         "2024": 2023, "2025": 2024},
      "observations_preserved_in_source": len(obs),
      "unique_wells_source": int(obs.gm_well_id.nunique()),
      "annual_rows": len(output_table),
      "annual_rows_by_crop_year": output_table.groupby("crop_year").size().to_dict(),
      "annual_unique_wells_by_crop_year": output_table.groupby("crop_year").gm_well_id.nunique().to_dict(),
      "source_versions": source_versions, "year_validations": qa_years,
      "gavin_comparison_shared_wells": int(comparison.well_id.nunique()),
      "gavin_comparison_rows": len(comparison),
      "gavin_comparison_all_match_after_3dp_rounding": bool(comparison.matches_rounding_tolerance.all()),
      "gavin_max_abs_difference": float(comparison.absolute_difference.max()),
      "input_sha256": hashes,
      "notes": [
          "pct_* crop class values are area divided by theoretical pi*r^2, preserving Gavin's denominator.",
          "dwr_polygon_footprint_pct uses actual buffer area and is polygon occupancy, not survey completeness.",
          "Area outside DWR polygons is unknown from this layer and must not be filled with zero crop.",
          "Within mapped polygons, class zero means no intersecting classified polygon of that group; it does not prove true absence outside mapped polygons.",
          "Conditional pct_*_of_mapped values use the union of DWR mapped polygon geometries as denominator.",
          "All buffers are complete 500/1000 m circles and are not clipped at county or HUC boundaries.",
          "DWR classes use Gavin's broad groups; T includes nurseries and YP includes young orchards/vines.",
          "Changes are percentage-point differences versus the prior crop year, not relative growth rates.",
          "Lag-1 is retrospective. Publication can postdate the nitrate measurement year.",
          "2024 source is provisional; DWR states the final release is planned for Q1 2027."
      ]}
(output / "validation.json").write_text(json.dumps(qa, indent=2, default=str), encoding="utf-8")
print(json.dumps({"output": str(output), "rows": len(output_table),
                  "columns": len(output_table.columns),
                  "gavin_wells": int(comparison.well_id.nunique()),
                  "gavin_rows": len(comparison), "checks": "passed"}, indent=2))

# Validate the existing shared CSV without overwriting it.
destination = (
    repo / "notebooks" / "crop_mapping"
    / "crop_features_lag1_2021_2024.csv"
)

if destination.exists():
    existing = pd.read_csv(destination, dtype={"gm_well_id": "string"})
    try:
        pd.testing.assert_frame_equal(
            existing, output_table, check_dtype=False, check_exact=False,
            atol=1e-10, rtol=1e-10,
        )
    except AssertionError as exc:
        raise ValueError(
            f"Existing crop CSV differs from this run and was not changed: {destination}"
        ) from exc
    print("Existing crop CSV matches; keeping it unchanged:", destination)
else:
    print("Shared crop CSV is absent; run output remains in:", csv_path)

{
  "output": "C:\\Users\\lomel\\Documents\\Data Science\\Data\\Erdos Project\\processed\\Crops\\annual_lag1_notebook_run_20261019_134020_20261019_134054",
  "rows": 5178,
  "columns": 109,
  "gavin_wells": 874,
  "gavin_rows": 15732,
  "checks": "passed"
}
Shared crop features saved: C:\Users\lomel\Documents\Data Science\GitHub\fall-2026-infrastructure-placement\notebooks\crop_mapping\crop_features_lag1_2021_2024.csv
